# Credit Risk Scorecard: A Real `sklearn.Pipeline` with LightGBM
### Cleaning → Imputation → Feature Engineering → LightGBM Modeling → Explainability — all inside one Pipeline object

This notebook builds an end-to-end credit risk scorecard on a 60,000-loan dataset with
43 raw features (28 bureau/application-style attributes matching the public **Lending Club**
schema, plus 15 synthetic cash-flow / alternative-data attributes in the style of
Plaid/Brigit-type bank-transaction data).

**The core business question**, mirroring real production work: *does adding cash-flow
data to a traditional bureau-only scorecard produce a meaningful, statistically validated
improvement in default prediction?*

We answer it with a proper **champion/challenger framework** — and critically, both models
are built as genuine **`sklearn.pipeline.Pipeline`** objects, not loose sequential code. Feature
engineering, imputation, and categorical encoding are all pipeline *steps*, which means:

- The exact same fitted pipeline object can be called as `pipeline.predict_proba(new_application)`
  on a single brand-new raw application at inference time — no manual preprocessing required
- There is zero risk of **training/serving skew** (a common real-world bug where the feature
  logic used in training silently drifts from what's used in production)
- The pipeline can be pickled, versioned, and deployed as a single artifact

> **Data note:** Bureau/application features are synthetically generated to match the real,
> publicly available Lending Club loan dataset schema and statistical relationships
> ([Kaggle: wordsforthewise/lending-club](https://www.kaggle.com/datasets/wordsforthewise/lending-club)).
> Cash-flow features (prefixed `cf_`) are fully synthetic — no public dataset contains
> real bank-transaction-level data, since that is proprietary, sensitive financial
> information. This is disclosed transparently in the project README.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import lightgbm as lgb

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, classification_report
from sklearn.inspection import permutation_importance

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", None)
RANDOM_STATE = 42

## 1. Load data and initial inspection

In [ ]:
df = pd.read_csv("../data/loan_data.csv", parse_dates=["issue_d"])
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

In [ ]:
missing = df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print("Columns with missing values:")
missing

In [ ]:
print(f"Charge-off rate: {df['charged_off'].mean():.2%}")
print(f"First payment default rate: {df['first_payment_default'].mean():.2%}")

## 2. Pre-split preparation

A few derived fields need to exist before we can even define the feature set: parsing the
messy `emp_length` string field, and computing the FICO midpoint from the real LC
`fico_range_low`/`fico_range_high` fields. These are dataset-level cleanup steps (true of
every row, not learned from training data), so they happen once here rather than inside
the pipeline.

In [ ]:
def parse_emp_length(val):
    if pd.isna(val):
        return np.nan
    if "< 1" in val:
        return 0
    if "10+" in val:
        return 10
    return int(val.split()[0])

df["emp_length_years"] = df["emp_length"].apply(parse_emp_length)
df["fico_mid"] = (df["fico_range_low"] + df["fico_range_high"]) / 2

# Outlier capping on annual_inc (a handful of obvious data-entry errors)
q99 = df["annual_inc"].quantile(0.99)
n_capped = (df["annual_inc"] > q99).sum()
df["annual_inc"] = df["annual_inc"].clip(upper=q99)
print(f"Capped {n_capped} extreme annual_inc values at 99th percentile (${q99:,.0f})")

## 3. Train / test split

We split **before** building the pipeline so that everything downstream — imputation
statistics, encoder categories, feature scaling if any were used — is fit only on the
training set and applied unchanged to test, avoiding leakage.

In [ ]:
TARGET = "charged_off"

drop_cols = [
    "loan_id", "issue_d", "fico_range_low", "fico_range_high", "emp_length",
    "first_payment_default", "charged_off",
    "grade",  # dropped: LC assigns grade using a model AFTER risk is assessed - using it
              # as a feature would leak the outcome we're trying to predict
]
feature_cols = [c for c in df.columns if c not in drop_cols]

X = df[feature_cols].copy()
y = df[TARGET].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)
print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")
print(f"Raw feature count: {len(feature_cols)}")

## 4. Define the pipeline components

### 4a. A custom `FeatureEngineer` transformer

This is the key piece that makes this a genuine pipeline rather than ad-hoc script code.
By writing feature engineering as a proper `sklearn` transformer (with `fit`/`transform`),
it becomes a **pipeline step** — meaning the exact same logic that ran at training time
automatically runs identically on any new data passed to `pipeline.predict_proba()`,
including a single new loan application at inference time. This eliminates a common
production bug: feature logic silently drifting between training code and serving code.

In [ ]:
class FeatureEngineer(BaseEstimator, TransformerMixin):
    """Adds ratio/interaction features on top of the raw input columns.

    Implemented as a proper sklearn transformer (fit/transform) so it can live
    inside a Pipeline. The transform is stateless (no parameters learned from
    training data), but living inside the Pipeline still matters: it guarantees
    this exact logic is what runs at inference time on brand-new applications,
    with zero risk of train/serve skew.
    """

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X["installment_to_income"] = X["installment"] * 12 / (X["annual_inc"] + 1)
        X["loan_to_income"] = X["loan_amnt"] / (X["annual_inc"] + 1)
        X["revol_bal_to_income"] = X["revol_bal"] / (X["annual_inc"] + 1)
        X["credit_file_depth"] = X["total_acc"] / (X["mths_since_earliest_cr_line"] / 12 + 1)
        X["high_util_flag"] = (X["revol_util"] > 80).astype(int)
        X["pay_freq_monthly_flag"] = (X["pay_frequency"] == "monthly").astype(int)
        if "cf_essential_spend_ratio" in X.columns:
            X["cf_net_cash_flow_margin"] = (
                1 - X["cf_essential_spend_ratio"] - X["cf_discretionary_spend_ratio"]
            )
            X["cf_balance_to_income_ratio"] = X["cf_avg_checking_balance"] * 12 / (X["annual_inc"] + 1)
            X["cf_distress_score"] = (
                X["cf_nsf_count_6mo"] + X["cf_overdraft_count_6mo"]
                + X["cf_negative_balance_days_90d"] / 10
            )
        return X

### 4b. A reusable pipeline-builder function

`build_credit_risk_pipeline()` assembles the full pipeline: `FeatureEngineer` →
`ColumnTransformer` (median imputation for numeric columns, explicit `"Missing"` category
+ ordinal encoding for categoricals) → `LGBMClassifier`. We call this function twice — once
for the bureau-only **champion** feature set, once for the bureau + cash-flow **challenger**
feature set — so both models are genuine, independently-fittable `Pipeline` objects built
from the same reusable construction logic.

In [ ]:
def build_credit_risk_pipeline(numeric_cols, categorical_cols, model_params):
    """Builds the full preprocessing + modeling pipeline as a single
    sklearn.pipeline.Pipeline object.

    Step order: feature engineering runs FIRST on the raw (unimputed) columns -
    the ratio features naturally inherit NaNs where their numerator/denominator
    inputs are missing - then the ColumnTransformer imputes everything and
    encodes categoricals, and finally LightGBM is fit on the processed matrix.
    """
    engineered_numeric = [
        "installment_to_income", "loan_to_income", "revol_bal_to_income",
        "credit_file_depth", "high_util_flag", "pay_freq_monthly_flag",
    ]
    if any(c.startswith("cf_") for c in numeric_cols):
        engineered_numeric += [
            "cf_net_cash_flow_margin", "cf_balance_to_income_ratio", "cf_distress_score"
        ]

    preprocessing = ColumnTransformer(transformers=[
        ("numeric", SimpleImputer(strategy="median"), numeric_cols + engineered_numeric),
        ("categorical", Pipeline([
            ("impute", SimpleImputer(strategy="constant", fill_value="Missing")),
            ("encode", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
        ]), categorical_cols),
    ])

    return Pipeline(steps=[
        ("feature_engineering", FeatureEngineer()),
        ("preprocessing", preprocessing),
        ("model", lgb.LGBMClassifier(**model_params)),
    ])

## 5. Champion / Challenger: two independently-fit pipelines

**Champion** — bureau and application attributes only (the traditional underwriting model)
**Challenger** — champion features **plus** the 15 cash-flow (`cf_`) attributes

Each is its own complete `Pipeline` — fit, predict, and inspect independently. This is the
exact framework used in production to decide whether a new data source justifies the cost
and complexity of integrating it into a live scorecard.

In [ ]:
lgb_params = dict(
    n_estimators=400, max_depth=5, num_leaves=31, learning_rate=0.04,
    subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_STATE, verbose=-1,
)

# --- CHAMPION pipeline: bureau-only columns ---
cf_cols = [c for c in X_train.columns if c.startswith("cf_")]
champion_X_train = X_train.drop(columns=cf_cols)
champion_X_test = X_test.drop(columns=cf_cols)

champion_numeric = champion_X_train.select_dtypes(include=[np.number]).columns.tolist()
champion_categorical = champion_X_train.select_dtypes(exclude=[np.number]).columns.tolist()

champion_pipeline = build_credit_risk_pipeline(champion_numeric, champion_categorical, lgb_params)
champion_pipeline.fit(champion_X_train, y_train)
champion_pred = champion_pipeline.predict_proba(champion_X_test)[:, 1]

print(f"Champion pipeline steps: {[s[0] for s in champion_pipeline.steps]}")
print(f"Champion feature count (pre-engineering): {len(champion_numeric) + len(champion_categorical)}")

In [ ]:
# --- CHALLENGER pipeline: full column set including cash-flow ---
challenger_numeric = X_train.select_dtypes(include=[np.number]).columns.tolist()
challenger_categorical = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

challenger_pipeline = build_credit_risk_pipeline(challenger_numeric, challenger_categorical, lgb_params)
challenger_pipeline.fit(X_train, y_train)
challenger_pred = challenger_pipeline.predict_proba(X_test)[:, 1]

print(f"Challenger pipeline steps: {[s[0] for s in challenger_pipeline.steps]}")
print(f"Challenger feature count (pre-engineering): {len(challenger_numeric) + len(challenger_categorical)}")
print(f"  (+{len(cf_cols)} cash-flow attributes vs. champion)")

## 6. Evaluate: AUC, Gini, KS

In [ ]:
def gini(auc):
    return 2 * auc - 1

def ks_statistic(y_true, scores):
    fpr, tpr, _ = roc_curve(y_true, scores)
    return np.max(np.abs(tpr - fpr))

champion_auc = roc_auc_score(y_test, champion_pred)
challenger_auc = roc_auc_score(y_test, challenger_pred)

results = pd.DataFrame({
    "Model": ["Champion (bureau only)", "Challenger (+ cash-flow)"],
    "AUC": [champion_auc, challenger_auc],
    "Gini": [gini(champion_auc), gini(challenger_auc)],
    "KS": [ks_statistic(y_test, champion_pred), ks_statistic(y_test, challenger_pred)],
})
results["Gini_lift_vs_champion"] = results["Gini"] - results.loc[0, "Gini"]
results

In [ ]:
fpr_champ, tpr_champ, _ = roc_curve(y_test, champion_pred)
fpr_chal, tpr_chal, _ = roc_curve(y_test, challenger_pred)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr_champ, tpr_champ, label=f"Champion: bureau only (AUC={champion_auc:.3f})")
ax.plot(fpr_chal, tpr_chal, label=f"Challenger: + cash-flow (AUC={challenger_auc:.3f})")
ax.plot([0, 1], [0, 1], "--", color="gray", label="Random")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve: Champion vs. Challenger Pipeline")
ax.legend()
plt.tight_layout()
plt.savefig("../outputs/roc_curve.png", dpi=120)
plt.show()

## 7. Feature importance & explainability

Permutation importance — a model-agnostic technique measuring the drop in pipeline
performance when a feature's values are randomly shuffled — works seamlessly here on
the **entire fitted pipeline object** `challenger_pipeline`, not just the bare model. This
correctly accounts for everything happening inside the pipeline (feature engineering,
imputation, encoding) when assessing each raw input column's true contribution.

In [ ]:
perm_result = permutation_importance(
    challenger_pipeline, X_test, y_test, n_repeats=10, random_state=RANDOM_STATE,
    scoring="roc_auc", n_jobs=-1
)

importance_df = pd.DataFrame({
    "feature": X_test.columns,
    "importance_mean": perm_result.importances_mean,
    "importance_std": perm_result.importances_std,
}).sort_values("importance_mean", ascending=False).reset_index(drop=True)

importance_df.head(15)

In [ ]:
top_n = 15
top_features = importance_df.head(top_n)

fig, ax = plt.subplots(figsize=(8, 6))
colors = ["#C44E52" if f.startswith("cf_") else "#4C72B0" for f in top_features["feature"]]
ax.barh(top_features["feature"], top_features["importance_mean"],
        xerr=top_features["importance_std"], color=colors)
ax.invert_yaxis()
ax.set_xlabel("Mean decrease in ROC-AUC when shuffled")
ax.set_title(f"Top {top_n} Feature Importances (red = cash-flow attribute)")
plt.tight_layout()
plt.savefig("../outputs/feature_importance.png", dpi=120)
plt.show()

n_cf_in_top = top_features["feature"].str.startswith("cf_").sum()
print(f"{n_cf_in_top} of the top {top_n} features are cash-flow attributes")

## 8. Model validation: confusion matrix & classification report

In [ ]:
cutoff = y_test.mean()
challenger_class = (challenger_pred >= cutoff).astype(int)

cm = confusion_matrix(y_test, challenger_class)
print(f"Confusion matrix at cutoff={cutoff:.3f}:")
print(cm)
print()
print(classification_report(y_test, challenger_class, target_names=["Good", "Charged Off"]))

## 9. The pipeline as a single deployable artifact

The real payoff of building this as a proper `Pipeline`: a single object handles raw input
all the way to a prediction. No manual preprocessing required at serving time — this is
exactly how the same code would be deployed in production (e.g., wrapped in an API endpoint
or batch scoring job).

In [ ]:
# Score one brand-new application using nothing but the raw columns -
# the pipeline handles feature engineering, imputation, and encoding internally
sample_application = X_test.iloc[[0]]
predicted_default_prob = challenger_pipeline.predict_proba(sample_application)[:, 1][0]

print(f"Raw input columns provided: {sample_application.shape[1]}")
print(f"Predicted default probability: {predicted_default_prob:.4f}")
print(f"Actual outcome: {'Charged Off' if y_test.iloc[0] == 1 else 'Good'}")

In [ ]:
champion_gini = results.loc[0, "Gini"]
challenger_gini = results.loc[1, "Gini"]
gini_lift = challenger_gini - champion_gini
n_cf_in_top15 = importance_df.head(15)["feature"].str.startswith("cf_").sum()

summary = f"""
BUSINESS SUMMARY
{'=' * 70}
Adding 15 synthetic cash-flow / alternative-data attributes to a traditional
bureau-only scorecard improved the Gini coefficient from {champion_gini:.3f} to
{challenger_gini:.3f} (a lift of {gini_lift:+.3f}), with {n_cf_in_top15} of the top 15
most predictive features coming from the cash-flow layer.

Both models were built as genuine sklearn.pipeline.Pipeline objects, not ad-hoc
sequential code - feature engineering, imputation, and encoding are pipeline steps,
which means the fitted pipeline can be called directly on a single new raw
application (pipeline.predict_proba(new_app)) with zero manual preprocessing and
zero risk of training/serving skew.

This directly mirrors a real production exercise: evaluating whether a new
alternative data source (e.g., Plaid/Brigit-style cash-flow attributes) justifies
the cost and complexity of integrating it into a live underwriting scorecard, built
the same way it would actually be deployed.
"""
print(summary)